
# Notebook 16 — Harmonized Clinical Model for External Validation

## هدف الدفتر

إعادة تدريب نموذج سريري داخل **PPMI** باستخدام المتغيرات السبعة المتوافقة فعليًا في **PPMI وPDBP**، ثم حفظ نموذج مقفل وجاهز للتطبيق لاحقًا على مجموعة التحقق الخارجي في PDBP.

### المتغيرات المشتركة النهائية

1. `ENROLL_AGE`
2. `baseline_NP3TOT`
3. `derived_years_since_PD_diagnosis`
4. `part2_NP2PTOT`
5. `part1_NP1RTOT`
6. `moca_MCATOT`
7. `baseline_NHY`

> لا تُضاف متغيرات جديدة بعد النظر إلى نتائج PDBP.  
> تعريف النتيجة يبقى مطابقًا لمشروع PPMI: `rapid_progression_q75`.


In [ ]:

# Cell 1 — Mount Google Drive and define project paths
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/PPMI_PD_Progression')
OUTPUT_DIR = PROJECT_DIR / 'outputs' / 'notebook_16_harmonized_clinical_model_7_features'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR:', PROJECT_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Project exists:', PROJECT_DIR.exists())


In [ ]:

# Cell 2 — Imports
import json
import warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, brier_score_loss
)
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

RANDOM_STATE = 42


In [ ]:

# Cell 3 — Utility functions
def find_first_existing(candidates):
    for path in candidates:
        path = Path(path)
        if path.exists():
            return path
    return None

def find_csv_by_keywords(base_dir, keyword_groups):
    base_dir = Path(base_dir)
    files = list(base_dir.rglob('*.csv'))
    for keywords in keyword_groups:
        keywords = [k.lower() for k in keywords]
        for file in files:
            name = file.name.lower()
            if all(k in name for k in keywords):
                return file
    return None

def binary_metrics(y_true, y_prob, threshold=0.50):
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        'ROC_AUC': roc_auc_score(y_true, y_prob),
        'PR_AUC': average_precision_score(y_true, y_prob),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'accuracy': accuracy_score(y_true, y_pred),
        'sensitivity': recall_score(y_true, y_pred, zero_division=0),
        'specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'Brier_score': brier_score_loss(y_true, y_prob),
        'threshold': threshold,
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)
    }


In [ ]:

# Cell 4 — Locate the PPMI analytic feature matrix
candidate_paths = [
    PROJECT_DIR / 'outputs' / 'notebook_03_baseline_predictors' / '08_feature_matrix_predictors_missingness_le_30pct.csv',
    PROJECT_DIR / 'outputs' / 'notebook_03_baseline_predictors' / '08_feature_matrix_predictors_missingness_le_30pct.csv',
    PROJECT_DIR / 'outputs' / 'notebook_04_preprocessing' / '06_train_raw_split_before_preprocessing.csv',
]

SOURCE_PATH = find_first_existing(candidate_paths)

if SOURCE_PATH is None:
    SOURCE_PATH = find_csv_by_keywords(
        PROJECT_DIR / 'outputs',
        [
            ['feature_matrix', 'missingness'],
            ['predictors', 'missingness'],
            ['raw_split', 'before_preprocessing']
        ]
    )

if SOURCE_PATH is None:
    raise FileNotFoundError(
        'Could not locate the PPMI analytic feature matrix. '
        'Expected an output from Notebook 03 or Notebook 04.'
    )

print('Detected source file:', SOURCE_PATH)
df_raw = pd.read_csv(SOURCE_PATH)
print('Shape:', df_raw.shape)
display(df_raw.head())
print('\nColumns:')
print(df_raw.columns.tolist())


In [ ]:

# Cell 5 — Identify ID and outcome columns
id_candidates = ['PATNO', 'participant_id', 'ID', 'id']
outcome_candidates = [
    'rapid_progression_q75',
    'rapid_progression',
    'outcome',
    'target'
]

id_col = next((c for c in id_candidates if c in df_raw.columns), None)
outcome_col = next((c for c in outcome_candidates if c in df_raw.columns), None)

if outcome_col is None:
    binary_cols = []
    for c in df_raw.columns:
        vals = set(pd.Series(df_raw[c]).dropna().astype(str).str.lower().unique())
        if vals and vals.issubset({'0', '1', '0.0', '1.0', 'false', 'true'}):
            binary_cols.append(c)
    print('Possible binary outcome columns:', binary_cols)
    if len(binary_cols) == 1:
        outcome_col = binary_cols[0]

assert outcome_col is not None, (
    'Outcome column not detected. Check the source file and set outcome_col manually.'
)

print('ID column:', id_col)
print('Outcome column:', outcome_col)
print(df_raw[outcome_col].value_counts(dropna=False))


In [ ]:

# Cell 6 — Define the final 7 harmonized clinical predictors
HARMONIZED_FEATURES = [
    'ENROLL_AGE',
    'baseline_NP3TOT',
    'derived_years_since_PD_diagnosis',
    'part2_NP2PTOT',
    'part1_NP1RTOT',
    'moca_MCATOT',
    'baseline_NHY'
]

availability = pd.DataFrame({
    'predictor': HARMONIZED_FEATURES,
    'available_in_ppmi_source': [c in df_raw.columns for c in HARMONIZED_FEATURES],
    'planned_pdbp_mapping': [
        'Demographics.age_at_baseline',
        'MDS_UPDRS_Part_III.mds_updrs_part_iii_summary_score',
        'Demographics.age_at_baseline - PD_Medical_History.age_at_diagnosis',
        'MDS_UPDRS_Part_II.mds_updrs_part_ii_summary_score',
        'MDS_UPDRS_Part_I.mds_updrs_part_i_summary_score',
        'MOCA.moca_total_score',
        'MDS_UPDRS_Part_III.code_upd2hy_hoehn_and_yahr_stage'
    ]
})

display(availability)
availability.to_csv(OUTPUT_DIR / '01_harmonized_feature_mapping.csv', index=False)

missing_features = availability.loc[
    ~availability['available_in_ppmi_source'], 'predictor'
].tolist()

if missing_features:
    raise KeyError(
        f'These harmonized predictors are absent from the detected PPMI file: {missing_features}'
    )


In [ ]:

# Cell 7 — Build the harmonized PPMI modeling dataset
keep_cols = ([id_col] if id_col else []) + HARMONIZED_FEATURES + [outcome_col]
df = df_raw[keep_cols].copy()

# Ensure binary outcome
df[outcome_col] = (
    df[outcome_col]
    .replace({'True': 1, 'False': 0, True: 1, False: 0})
)
df[outcome_col] = pd.to_numeric(df[outcome_col], errors='coerce')

# Remove rows without outcome
df = df.dropna(subset=[outcome_col]).copy()
df[outcome_col] = df[outcome_col].astype(int)

# Convert predictor types
numeric_features = [
    'ENROLL_AGE',
    'baseline_NP3TOT',
    'derived_years_since_PD_diagnosis',
    'part2_NP2PTOT',
    'part1_NP1RTOT',
    'moca_MCATOT'
]
categorical_features = ['baseline_NHY']

for c in numeric_features:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['baseline_NHY'] = df['baseline_NHY'].astype('string')

print('Final harmonized cohort shape:', df.shape)
print('\nOutcome distribution:')
display(df[outcome_col].value_counts().rename_axis('class').reset_index(name='n'))

missingness = (
    df[HARMONIZED_FEATURES]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename('missing_percent')
    .reset_index()
    .rename(columns={'index': 'predictor'})
)
display(missingness)
missingness.to_csv(
    OUTPUT_DIR / '02_harmonized_predictor_missingness.csv',
    index=False
)


In [ ]:

# Cell 8 — Leakage-safe train/test split
X = df[HARMONIZED_FEATURES].copy()
y = df[outcome_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

split_summary = pd.DataFrame([
    {
        'split': 'train',
        'n': len(y_train),
        'rapid_progressors': int(y_train.sum()),
        'rapid_progressor_percent': 100 * y_train.mean()
    },
    {
        'split': 'test',
        'n': len(y_test),
        'rapid_progressors': int(y_test.sum()),
        'rapid_progressor_percent': 100 * y_test.mean()
    }
])

display(split_summary)
split_summary.to_csv(OUTPUT_DIR / '03_train_test_split_summary.csv', index=False)


In [ ]:

# Cell 9 — Preprocessing pipeline
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('continuous', numeric_transformer, numeric_features),
    ('categorical', categorical_transformer, categorical_features)
], remainder='drop')

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)


In [ ]:

# Cell 10 — Candidate harmonized models
models = {
    'Logistic_L2_balanced': LogisticRegression(
        penalty='l2',
        C=1.0,
        class_weight='balanced',
        solver='liblinear',
        max_iter=5000,
        random_state=RANDOM_STATE
    ),
    'Logistic_ElasticNet_balanced': LogisticRegression(
        penalty='elasticnet',
        l1_ratio=0.5,
        C=1.0,
        class_weight='balanced',
        solver='saga',
        max_iter=5000,
        random_state=RANDOM_STATE
    ),
    'RandomForest_balanced': RandomForestClassifier(
        n_estimators=500,
        class_weight='balanced',
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    'HistGradientBoosting': HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=RANDOM_STATE
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    'ROC_AUC': 'roc_auc',
    'PR_AUC': 'average_precision',
    'balanced_accuracy': 'balanced_accuracy',
    'F1': 'f1'
}

cv_rows = []
pipelines = {}

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('model', model)
    ])
    pipelines[name] = pipe

    scores = cross_validate(
        pipe, X_train, y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )

    cv_rows.append({
        'model': name,
        'CV_ROC_AUC_mean': scores['test_ROC_AUC'].mean(),
        'CV_ROC_AUC_sd': scores['test_ROC_AUC'].std(),
        'CV_PR_AUC_mean': scores['test_PR_AUC'].mean(),
        'CV_balanced_accuracy_mean': scores['test_balanced_accuracy'].mean(),
        'CV_F1_mean': scores['test_F1'].mean()
    })

cv_results = pd.DataFrame(cv_rows).sort_values(
    ['CV_ROC_AUC_mean', 'CV_PR_AUC_mean'],
    ascending=False
).reset_index(drop=True)

display(cv_results)
cv_results.to_csv(OUTPUT_DIR / '04_harmonized_cv_model_comparison.csv', index=False)


In [ ]:

# Cell 11 — Select the best model using training CV only
best_model_name = cv_results.iloc[0]['model']
best_pipeline = clone(pipelines[best_model_name])

print('Selected model:', best_model_name)

best_pipeline.fit(X_train, y_train)
test_prob = best_pipeline.predict_proba(X_test)[:, 1]

test_metrics = binary_metrics(y_test, test_prob, threshold=0.50)
test_metrics['model'] = best_model_name
test_metrics_df = pd.DataFrame([test_metrics])

display(test_metrics_df)
test_metrics_df.to_csv(
    OUTPUT_DIR / '05_harmonized_internal_test_performance.csv',
    index=False
)


In [ ]:

# Cell 12 — Threshold screening on the internal held-out test set
threshold_rows = []

for threshold in np.arange(0.10, 0.81, 0.05):
    row = binary_metrics(y_test, test_prob, threshold=float(threshold))
    threshold_rows.append(row)

threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table)

threshold_table.to_csv(
    OUTPUT_DIR / '06_harmonized_threshold_screening_internal_test.csv',
    index=False
)

# Threshold chosen by maximum balanced accuracy, documented for secondary analysis.
selected_threshold = float(
    threshold_table.sort_values(
        ['balanced_accuracy', 'F1'],
        ascending=False
    ).iloc[0]['threshold']
)

print('Secondary operating threshold:', selected_threshold)


In [ ]:

# Cell 13 — Sensitivity analysis without baseline MDS-UPDRS III
SENSITIVITY_FEATURES = [
    c for c in HARMONIZED_FEATURES if c != 'baseline_NP3TOT'
]

sens_numeric = [
    c for c in numeric_features if c != 'baseline_NP3TOT'
]
sens_categorical = categorical_features.copy()

sens_preprocessor = ColumnTransformer([
    ('continuous', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), sens_numeric),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), sens_categorical)
])

sens_pipe = Pipeline([
    ('preprocessor', sens_preprocessor),
    ('model', clone(models[best_model_name]))
])

sens_pipe.fit(X_train[SENSITIVITY_FEATURES], y_train)
sens_prob = sens_pipe.predict_proba(
    X_test[SENSITIVITY_FEATURES]
)[:, 1]

sens_metrics = binary_metrics(
    y_test, sens_prob, threshold=0.50
)
sens_metrics['model'] = best_model_name
sens_metrics['feature_set'] = (
    'harmonized_without_baseline_NP3TOT'
)

sens_metrics_df = pd.DataFrame([sens_metrics])
display(sens_metrics_df)

sens_metrics_df.to_csv(
    OUTPUT_DIR / '07_sensitivity_without_baseline_NP3TOT.csv',
    index=False
)


In [ ]:

# Cell 14 — Fit and lock final harmonized models on the full PPMI cohort
# The external PDBP data must not be used for tuning or model selection.

final_primary_pipeline = clone(pipelines[best_model_name])
final_primary_pipeline.fit(X, y)

final_sensitivity_pipeline = clone(sens_pipe)
final_sensitivity_pipeline.fit(X[SENSITIVITY_FEATURES], y)

joblib.dump(
    final_primary_pipeline,
    OUTPUT_DIR / 'harmonized_primary_model_ppmi_7_features.joblib'
)
joblib.dump(
    final_sensitivity_pipeline,
    OUTPUT_DIR / 'harmonized_sensitivity_model_without_baseline_NP3TOT_6_features.joblib'
)

pd.DataFrame({'predictor': HARMONIZED_FEATURES}).to_csv(
    OUTPUT_DIR / '08_harmonized_primary_feature_list.csv',
    index=False
)
pd.DataFrame({'predictor': SENSITIVITY_FEATURES}).to_csv(
    OUTPUT_DIR / '09_harmonized_sensitivity_feature_list.csv',
    index=False
)

print('Final model packages saved.')


In [ ]:

# Cell 15 — Save processed feature names and model metadata
processed_feature_names = (
    final_primary_pipeline
    .named_steps['preprocessor']
    .get_feature_names_out()
    .tolist()
)

pd.DataFrame({
    'processed_feature_name': processed_feature_names
}).to_csv(
    OUTPUT_DIR / '10_harmonized_processed_feature_names.csv',
    index=False
)

metadata = {
    'notebook': 'Notebook 16 — Harmonized Clinical Model',
    'development_dataset': 'PPMI',
    'planned_external_validation_dataset': 'PDBP',
    'outcome_column': outcome_col,
    'outcome_definition': (
        'Rapid motor progression using the locked PPMI threshold: '
        'annualized change in MDS-UPDRS Part III >= 5.0793 points/year'
    ),
    'primary_features': HARMONIZED_FEATURES,
    'sensitivity_features': SENSITIVITY_FEATURES,
    'selected_model': best_model_name,
    'random_state': RANDOM_STATE,
    'internal_test_threshold_primary': 0.50,
    'secondary_operating_threshold': selected_threshold,
    'external_validation_rule': (
        'No retraining, feature selection, threshold optimization, '
        'or hyperparameter tuning using PDBP.'
    )
}

with open(OUTPUT_DIR / '11_harmonized_model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))


In [ ]:

# Cell 16 — Quality-control checklist and summary report
qc = []

def add_qc(check, status, detail):
    qc.append({'check': check, 'status': status, 'detail': detail})

add_qc(
    'PPMI source file detected',
    'PASS' if SOURCE_PATH.exists() else 'FAIL',
    str(SOURCE_PATH)
)
add_qc(
    'All harmonized predictors available',
    'PASS' if not missing_features else 'FAIL',
    f'{len(HARMONIZED_FEATURES) - len(missing_features)}/{len(HARMONIZED_FEATURES)}'
)
add_qc(
    'Outcome is binary',
    'PASS' if set(y.unique()).issubset({0, 1}) else 'FAIL',
    str(sorted(y.unique().tolist()))
)
add_qc(
    'Stratified train/test split preserved',
    'PASS',
    f'train={len(y_train)}, test={len(y_test)}'
)
add_qc(
    'Primary model saved',
    'PASS' if (OUTPUT_DIR / 'harmonized_primary_model_ppmi_7_features.joblib').exists() else 'FAIL',
    'harmonized_primary_model_ppmi_7_features.joblib'
)
add_qc(
    'Sensitivity model saved',
    'PASS' if (OUTPUT_DIR / 'harmonized_sensitivity_model_without_baseline_NP3TOT_6_features.joblib').exists() else 'FAIL',
    'harmonized_sensitivity_model_without_baseline_NP3TOT_6_features.joblib'
)
add_qc(
    'PDBP kept completely external',
    'PASS',
    'No PDBP participant-level data were used in this notebook.'
)

qc_df = pd.DataFrame(qc)
display(qc_df)
qc_df.to_csv(OUTPUT_DIR / '12_quality_control_checklist.csv', index=False)

summary_lines = [
    'Notebook 16 — Harmonized Clinical Model',
    '',
    f'PPMI source: {SOURCE_PATH}',
    f'PPMI harmonized cohort: {len(df)}',
    f'Rapid progressors: {int(y.sum())} ({100*y.mean():.2f}%)',
    f'Harmonized raw predictors: {len(HARMONIZED_FEATURES)}',
    f'Selected model: {best_model_name}',
    f'Internal held-out ROC-AUC: {test_metrics["ROC_AUC"]:.4f}',
    f'Internal held-out PR-AUC: {test_metrics["PR_AUC"]:.4f}',
    f'Internal held-out balanced accuracy: {test_metrics["balanced_accuracy"]:.4f}',
    f'Secondary operating threshold: {selected_threshold:.2f}',
    '',
    'Saved model:',
    str(OUTPUT_DIR / 'harmonized_primary_model_ppmi_7_features.joblib'),
    '',
    'Next step:',
    'Build the PDBP harmonized external-validation matrix with identical raw columns,',
    'load this saved pipeline, and evaluate it without retraining or threshold tuning.'
]

summary_text = '\n'.join(summary_lines)
print(summary_text)

with open(OUTPUT_DIR / '13_notebook_16_summary_report.txt', 'w') as f:
    f.write(summary_text)


In [ ]:

# Final validation — confirm feature consistency before running all cells
expected_primary = {
    'ENROLL_AGE',
    'baseline_NP3TOT',
    'derived_years_since_PD_diagnosis',
    'part2_NP2PTOT',
    'part1_NP1RTOT',
    'moca_MCATOT',
    'baseline_NHY'
}

assert set(HARMONIZED_FEATURES) == expected_primary
assert 'part1p_NP1PTOT' not in HARMONIZED_FEATURES
assert len(HARMONIZED_FEATURES) == 7

print('Feature consistency check: PASS')
print('Primary harmonized predictors:', HARMONIZED_FEATURES)



## المخرجات المتوقعة

سيُنشئ الدفتر داخل:

```text
PPMI_PD_Progression/outputs/notebook_16_harmonized_clinical_model/
```

أهم الملفات:

- `harmonized_primary_model_ppmi.joblib`
- `harmonized_sensitivity_model_without_baseline_NP3TOT.joblib`
- `08_harmonized_primary_feature_list.csv`
- `10_harmonized_processed_feature_names.csv`
- `11_harmonized_model_metadata.json`
- `12_quality_control_checklist.csv`
- `13_notebook_16_summary_report.txt`

### الخطوة التالية

إنشاء **Notebook 17** لبناء مصفوفة PDBP المتوافقة وتطبيق النموذج المحفوظ دون إعادة تدريب.
